# Cryptographic Faithful Aggregation on Edge-IIoTset - FINAL RUN (v5, single profile)

This notebook has **one configuration** and is meant to be executed once. It produces every number the paper needs:

* **Task:** 15-class attack detection on Edge-IIoTset (macro-F1), 10 sites with non-IID class mixes, **10 seeds**, 30 rounds.
* **Recommended design:** clean root-set reference, tolerant cosine threshold tau = -0.3, public norm bound kappa = 3.5 (x root-update norm).
* **Baselines:** local-only, centralized, FedAvg, trimmed mean, multi-Krum, secure norm-only, and the previous-aggregate reference (capture demo).
* **Attacks:** sign-flip, label-flip, targeted flip, backdoor, **boosted backdoor** (x3), adaptive (2 and 3 of 10 sites).
* **Mitigation arms** on top of the recommended design: clip, clip + noise, and clip + noise + aggregate clip at noise multipliers z = 0.5, 1, 2, each with an **RDP-accounted epsilon** (client-level, T rounds, full participation; accept bits are *not* covered by the accounting).
* **kappa sweep** (1.5, 2.5, 3.5, 5) and tau comparison.
* **Privacy attack** (no-attack runs, first 10 rounds): per-site class-mix inference, a relational-structure attack (full-update and output-bias similarity, nearest-neighbour hit rate), accept-bit leakage over **all** rounds, for plaintext updates vs one server's share vs colluding servers.
* **Cost benchmark** with real Pedersen commitments, Merkle proofs and audit.

**Run:** put this notebook, `faithful_agg.py` (the latest version) and `DNN-EdgeIIoT-dataset.csv` in one folder and run all cells. Expect roughly 1 to 1.5 hours. Progress is saved after every experiment; if it stops, re-run all cells and it resumes (delete the results folder if you change the config). At the end upload `results_pack.zip` (aggregates only, no raw data).

*Rigor notes:* secure comparison, range proofs, triple generation and secure clipping are ideal functionalities (costs are estimates); the Pedersen group is a toy 512-bit group (timings are upper bounds); the inference attacker is generic; epsilon covers only the Gaussian noise on the released aggregate.

In [ ]:
# ============================== 1. CONFIG ==============================
import os, sys, json, time, math, platform, zipfile, datetime, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

DATA_PATH = os.environ.get("EDGE_DATA", "../data/DNN-EdgeIIoT-dataset.csv")
CORE_DIR  = os.environ.get("CORE_DIR", "..")
OUT_DIR   = Path(os.environ.get("OUT_DIR", "../results/v5_full"))

CFG = dict(
    seeds=list(range(10)), rounds=30,
    scenarios=["none", "signflip", "labelflip", "targeted_flip", "backdoor", "backdoor_boost", "adaptive", "adaptive_f30"],
    n_sites=10, n_byz=2, dirichlet_alpha=0.5,
    hidden=24, max_features=80, use_payload_features=True,
    lr=0.1, batch=128, local_steps=30,
    n_normal=100_000, n_per_attack=10_000,
    root_size=1500, test_frac=0.2,
    krum_f=3, trim_k=3,
    kappa=3.5, kappa_sweep=[1.5, 2.5, 5.0],
    taus_prev=[0.0], taus_root=[-0.3, 0.0],
    backdoor_frac=0.5, trigger_k=3, trigger_val=4.0, boost=3.0,
    mitigation=True, mit_clip=1.0, mit_z_list=[0.5, 1.0, 2.0], mit_agg=3.0, dp_delta=1e-5,
    obs_rounds=10,
    arm_filter=None,
)
CFG.update(json.loads(os.environ.get("CFG_OVERRIDES", "{}")))        # CFG_OVERRIDES: smoke tests only
PROFILE = "full"

sys.path.insert(0, str(Path(CORE_DIR).resolve()))
import faithful_agg as fa
assert hasattr(fa, "ideal_accept_tolerant"), "faithful_agg.py is outdated: use the latest version"
OUT_DIR.mkdir(parents=True, exist_ok=True); (OUT_DIR / "figs").mkdir(exist_ok=True)
print("output:", OUT_DIR.resolve()); print(json.dumps(CFG, indent=1))

## 2. Data: chunked sampling, feature engineering, class mapping
Placeholder values (`0`, `0.0`, `nan`) in text columns mean "field absent". For text columns we keep: present-flag, log length, share of special characters, and signature flags (SQL / XSS / path-traversal / upload tokens). Identifier columns (IPs, timestamps, source ports) are dropped. Low-cardinality text columns are one-hot encoded; destination ports are one-hot (top 15) plus a high-port flag. The top `max_features` features by strongest one-vs-rest correlation with any class are kept.

In [ ]:
# ============================== 2. DATA ==============================
IDENT_DROP = {"frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
              "icmp.transmit_timestamp", "tcp.srcport"}
PORT_COLS = ["tcp.dstport", "udp.port"]
PLACEHOLDERS = {"0", "0.0", "nan", "NaN", "", "None", "nan.0"}
RX = {"sql": r"(?i)(select|union|insert|drop|update|delete|or\s+1=1|sleep\(|--|%27)",
      "xss": r"(?i)(<script|javascript:|onerror|onload|alert\(|%3cscript|<img|<svg)",
      "trav": r"(?i)(\.\./|%2e%2e|/etc/passwd|cmd\.exe)",
      "upl": r"(?i)(multipart|filename=|\.php|\.jsp|\.exe|content-disposition)"}

def find_col(cols, name):
    for c in cols:
        if c.strip().lower() == name.lower():
            return c
    return None

head = pd.read_csv(DATA_PATH, nrows=5, low_memory=False)
YCOL, TCOL = find_col(head.columns, "Attack_label"), find_col(head.columns, "Attack_type")
assert YCOL and TCOL, f"label columns not found; last columns: {list(head.columns[-4:])}"
lab = pd.read_csv(DATA_PATH, usecols=[YCOL, TCOL], low_memory=False)
lab[TCOL] = lab[TCOL].astype(str)
counts = lab.groupby(TCOL)[YCOL].agg(["size", "mean"])
normal_types = set(counts.index[counts["mean"] == 0])
cap = {t: (CFG["n_normal"] if t in normal_types else CFG["n_per_attack"]) for t in counts.index}
keep_p = {t: (1.0 if cap[t] is None else min(1.0, cap[t] / counts.loc[t, "size"])) for t in counts.index}
FILE_ROWS, FILE_COUNTS = int(len(lab)), counts["size"].astype(int).to_dict()
print(f"{FILE_ROWS:,} rows in file"); print(counts["size"].to_string()); del lab

rng_s = np.random.default_rng(12345); parts = []
for ch in pd.read_csv(DATA_PATH, chunksize=250_000, low_memory=False):
    p = ch[TCOL].astype(str).map(keep_p).fillna(0).values
    parts.append(ch[rng_s.random(len(ch)) < p])
raw = pd.concat(parts, ignore_index=True); del parts

types_arr = raw[TCOL].astype(str).values
types_arr = np.where(np.isin(types_arr, list(normal_types)), "Normal", types_arr)
CLASSES = sorted(np.unique(types_arr)); C = len(CLASSES)
NORMAL = CLASSES.index("Normal")
y_arr = np.array([CLASSES.index(t) for t in types_arr])
y_bin = (y_arr != NORMAL).astype(int)

feats, skipped, groups = {}, [], {"numeric": 0, "onehot": 0, "text": 0, "port": 0}
for c in [c for c in raw.columns if c not in (YCOL, TCOL) and c.strip() not in IDENT_DROP and c not in PORT_COLS]:
    try:
        s = raw[c]
        if pd.api.types.is_numeric_dtype(s):
            feats[c] = s.astype(np.float32); groups["numeric"] += 1; continue
        st = s.astype(str); pres = ~st.isin(PLACEHOLDERS); num = pd.to_numeric(st, errors="coerce")
        if float((pres & num.isna()).mean()) < 0.001:
            feats[c] = num.fillna(0).astype(np.float32); groups["numeric"] += 1; continue
        feats[c + "__present"] = pres.astype(np.float32); groups["text"] += 1
        if st.nunique() <= 20:
            for name, col in pd.get_dummies(st, prefix=c, dtype=np.float32).items():
                feats[name] = col; groups["onehot"] += 1
        if CFG["use_payload_features"]:
            L = st.str.len().where(pres, 0)
            feats[c + "__loglen"] = np.log1p(L).astype(np.float32)
            feats[c + "__special"] = (st.str.count(r"[^A-Za-z0-9 ]").where(pres, 0) / L.clip(lower=1)).astype(np.float32)
            for nm, rx in RX.items():
                feats[f"{c}__{nm}"] = (st.str.contains(rx, regex=True) & pres).astype(np.float32)
    except Exception as e:
        skipped.append((c, repr(e)[:80]))
for c in PORT_COLS:
    if c in raw.columns:
        try:
            v = pd.to_numeric(raw[c].astype(str), errors="coerce").fillna(0)
            for pv in v.value_counts().index[:15]:
                feats[f"{c}=={int(pv)}"] = (v == pv).astype(np.float32); groups["port"] += 1
            feats[c + "__high"] = (v >= 1024).astype(np.float32)
        except Exception as e:
            skipped.append((c, repr(e)[:80]))
Xdf = pd.DataFrame(feats).apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)
Xdf = Xdf.loc[:, Xdf.std() > 0]
del raw, feats

Xv = Xdf.values; Xz = (Xv - Xv.mean(0)) / (Xv.std(0) + 1e-6)
Yc = np.eye(C, dtype=np.float32)[y_arr]; Yc = (Yc - Yc.mean(0)) / (Yc.std(0) + 1e-6)
score = (np.abs(Xz.T @ Yc) / len(Xz)).max(1)
order_f = np.argsort(-score)[: CFG["max_features"]]
FEATURES = [Xdf.columns[i] for i in order_f]
X_all = Xv[:, order_f].astype(np.float32); del Xz, Xdf, Xv
print(f"\nclasses ({C}): {CLASSES}")
print(f"sampled rows {len(y_arr):,} | attack share {y_bin.mean():.1%} | features kept {len(FEATURES)} (groups before selection: {groups})")
if skipped: print("skipped columns:", skipped)
print("top features:", FEATURES[:15])

## 3. Model, attacks, aggregators
Softmax MLP (NumPy, one flat parameter vector). Public norm bound each round `B_t = kappa * ||root update||`. The root set is a small clean server-side capture (e.g. a testbed). Attacks (2 of 10 sites, 3 for `*_f30`):
* `signflip`: -4u. `labelflip`: labels rotated y -> y+1. `targeted_flip`: all attack classes relabelled Normal (evasion). `backdoor`: 50% of attack samples get a trigger on 3 low-importance features and are relabelled Normal; **ASR** = share of triggered attack samples classified Normal.
* `adaptive`: worst case. Attacker knows B, tau and the reference; submits norm 0.95B at the minimum allowed cosine to the reference, the rest pointing along the **loss-ascent direction** (-honest update).

In [ ]:
# ============================== 3. MODEL / ATTACKS / AGGREGATORS ==============================
F = X_all.shape[1]; H = CFG["hidden"]; D = F * H + H + H * C + C
TRIG = np.arange(F - CFG["trigger_k"], F)          # lowest-importance features (selection order is by score)
print(f"input features {F}, classes {C}, model dimension d = {D}")

def unpack(w):
    i = 0
    W1 = w[i:i + F * H].reshape(F, H); i += F * H
    b1 = w[i:i + H]; i += H
    W2 = w[i:i + H * C].reshape(H, C); i += H * C
    return W1, b1, W2, w[i:i + C]

def init_w(rng):
    return np.concatenate([rng.normal(0, np.sqrt(2 / F), (F, H)).ravel(), np.zeros(H),
                           rng.normal(0, np.sqrt(2 / H), (H, C)).ravel(), np.zeros(C)])

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def predict_class(w, Xm):
    W1, b1, W2, b2 = unpack(w)
    return (np.maximum(Xm @ W1 + b1, 0.0) @ W2 + b2).argmax(1)

def local_update(w, Xm, ym, rng, batch=None):
    w0 = w.copy(); w = w.copy(); n = len(ym); bs = batch or CFG["batch"]
    for _ in range(CFG["local_steps"]):
        idx = rng.integers(0, n, bs)
        xb = Xm[idx].astype(np.float64); yb = ym[idx]
        W1, b1, W2, b2 = unpack(w)
        z1 = xb @ W1 + b1; a1 = np.maximum(z1, 0.0)
        P = softmax(a1 @ W2 + b2); P[np.arange(len(yb)), yb] -= 1.0; P /= len(yb)
        dz1 = (P @ W2.T) * (z1 > 0)
        g = np.concatenate([(xb.T @ dz1).ravel(), dz1.sum(0), (a1.T @ P).ravel(), P.sum(0)])
        w -= CFG["lr"] * g
    return w - w0

def metrics(w, Xm, ym, full=False):
    pred = predict_class(w, Xm)
    cm = np.bincount(ym * C + pred, minlength=C * C).reshape(C, C).astype(float)
    tp = np.diag(cm); sup = cm.sum(1); ps = cm.sum(0)
    rec = tp / np.maximum(sup, 1); prec = tp / np.maximum(ps, 1)
    f1c = 2 * prec * rec / np.maximum(prec + rec, 1e-12); pres = sup > 0
    att = ym != NORMAL; pa = pred != NORMAL
    out = dict(acc=float((pred == ym).mean()), macro_f1=float(f1c[pres].mean()), bal_acc=float(rec[pres].mean()),
               detect=float((pa & att).sum() / max(att.sum(), 1)), fpr=float((pa & ~att).sum() / max((~att).sum(), 1)))
    if full:
        out["per_class_recall"] = {CLASSES[i]: round(float(rec[i]), 3) for i in range(C) if pres[i]}
    return out

def asr(w, Xtrig):
    return float((predict_class(w, Xtrig) == NORMAL).mean()) if len(Xtrig) else float("nan")

def agg_trimmed(U, k):
    return np.sort(U, axis=0)[k:len(U) - k].mean(0) if (k > 0 and len(U) > 2 * k) else U.mean(0)

def agg_multikrum(U, f):
    n = len(U); G = U @ U.T; sq = np.diag(G); D2 = sq[:, None] + sq[None, :] - 2 * G
    mc = max(1, n - f - 2)
    scores = [np.sort(np.delete(D2[i], i))[:mc].sum() for i in range(n)]
    sel = np.argsort(scores)[:max(1, n - f)]
    return U[sel].mean(0), set(int(s) for s in sel)

def adaptive_vec(B, ref, tau, u_honest):
    neg = -u_honest
    if ref is None or np.linalg.norm(ref) == 0:
        return 0.95 * B * neg / (np.linalg.norm(neg) + 1e-12)
    rh = ref / np.linalg.norm(ref)
    p = neg - (neg @ rh) * rh; p /= (np.linalg.norm(p) + 1e-12)
    c = min(max(tau if tau is not None else 0.0, 0.0) + 0.02, 0.999)
    return 0.95 * B * (c * rh + np.sqrt(1 - c * c) * p)

def dirichlet_partition(train_idx, rng, tries=300, min_size=300):
    classes = np.unique(types_arr[train_idx])
    for _ in range(tries):
        parts = [[] for _ in range(CFG["n_sites"])]
        for t in classes:
            idx = rng.permutation(train_idx[types_arr[train_idx] == t])
            pr = rng.dirichlet([CFG["dirichlet_alpha"]] * CFG["n_sites"])
            cuts = (np.cumsum(pr) * len(idx)).astype(int)[:-1]
            for s, chunk in enumerate(np.split(idx, cuts)):
                parts[s].extend(chunk.tolist())
        if all(len(p) >= min_size for p in parts) and all(0.05 <= y_bin[p].mean() <= 0.95 for p in parts):
            return [np.array(p) for p in parts]
    raise RuntimeError("partition constraints not met: raise dirichlet_alpha / lower min_size / use more rows")

def build_bundle(seed):
    rng = np.random.default_rng(1000 + seed); test, root, train = [], [], []
    for t in np.unique(types_arr):
        idx = rng.permutation(np.where(types_arr == t)[0])
        nt = max(1, int(round(len(idx) * CFG["test_frac"]))); test.append(idx[:nt]); rest = idx[nt:]
        nr = max(1, int(round(CFG["root_size"] * len(idx) / len(types_arr)))); root.append(rest[:nr]); train.append(rest[nr:])
    test, root, train = (np.concatenate(a) for a in (test, root, train))
    mu = X_all[train].mean(0); sd = X_all[train].std(0) + 1e-6
    Z = np.clip((X_all - mu) / sd, -5, 5).astype(np.float32)
    parts = dirichlet_partition(train, rng)
    rh = np.bincount(y_arr[root], minlength=C) / len(root)
    tv = [float(0.5 * np.abs(np.bincount(y_arr[p], minlength=C) / len(p) - rh).sum()) for p in parts]
    Xt, yt = Z[test], y_arr[test]
    Xtrig = Xt[yt != NORMAL].copy(); Xtrig[:, TRIG] = CFG["trigger_val"]
    return dict(sites=[(Z[p], y_arr[p]) for p in parts], root=(Z[root], y_arr[root]), test=(Xt, yt), test_trig=Xtrig,
                site_sizes=[len(p) for p in parts], site_attack_frac=[float(y_bin[p].mean()) for p in parts],
                site_n_classes=[int(len(np.unique(y_arr[p]))) for p in parts], site_tv=tv,
                site_hist=[(np.bincount(y_arr[p], minlength=C) / len(p)).tolist() for p in parts])

In [ ]:
# ---- honest update-norm ratios (documents the choice of kappa); kappa is fixed in CFG ----
b0 = build_bundle(0); rng = np.random.default_rng(99); w = init_w(rng); ratios = []
for r in range(6):
    ru = local_update(w, *b0["root"], rng); rn = np.linalg.norm(ru)
    ups = [local_update(w, Xs, ys, rng) for Xs, ys in b0["sites"]]
    ratios += [np.linalg.norm(u) / rn for u in ups]; w = w + np.mean(ups, axis=0)
pct = {f"p{q}": round(float(np.percentile(ratios, q)), 3) for q in (50, 90, 99, 100)}
KAPPA = float(CFG["kappa"]) if CFG["kappa"] is not None else max(2.0, math.ceil(1.25 * pct["p99"] * 2) / 2)
print("honest ||u||/||root|| percentiles:", pct, "-> kappa used:", KAPPA); del b0
CFG["kappa_used"] = KAPPA; CFG["honest_norm_ratio_pct"] = pct
json.dump(CFG, open(OUT_DIR / "config.json", "w"), indent=2)

## 4. Experiment loop (resumable)
Arms are defined once and run over their scenario lists. In no-attack runs an inference attacker observes the first `obs_rounds` rounds (plaintext views for FedAvg / Krum / trimmed mean; one server's share view plus accept bits for secure arms; colluding view also scored). Mitigation arms apply ideal secure clipping, Gaussian noise inside the aggregate, and a root-guided clip of the revealed aggregate.

In [ ]:
# ============================== 4. EXPERIMENT LOOP ==============================
ARMS = []
def add(name, fam, tau=None, scn=None, **kw):
    ARMS.append(dict(name=name, fam=fam, tau=tau, scn=scn, **kw))
have = lambda *names: [s for s in names if s in CFG["scenarios"]]
for n_, f_ in (("fedavg", "fedavg"), ("trimmed_mean", "trimmed"), ("multi_krum", "krum"), ("secure_norm_only", "norm_only")):
    add(n_, f_)
for t in CFG["taus_prev"]: add(f"faithful_prev_t{t}", "faithful_prev", t, scn=have("none", "adaptive", "adaptive_f30"))
for t in CFG["taus_root"]: add(f"faithful_root_t{t}", "faithful_root", t)
for kp in CFG["kappa_sweep"]: add(f"faithful_root_t-0.3_kappa{kp}", "faithful_root", -0.3, scn=have("none", "signflip", "adaptive_f30"), kappa=kp)
if CFG["mitigation"]:
    ms = have("none", "backdoor", "backdoor_boost", "adaptive_f30", "signflip")
    add("faithful_root_t-0.3_clip", "faithful_root", -0.3, scn=ms, clip=CFG["mit_clip"])
    z0 = CFG["mit_z_list"][0]
    add(f"faithful_root_t-0.3_clipnoise_z{z0}", "faithful_root", -0.3, scn=ms, clip=CFG["mit_clip"], z=z0)
    for z in CFG["mit_z_list"]:
        add(f"faithful_root_t-0.3_all_z{z}", "faithful_root", -0.3, scn=ms, clip=CFG["mit_clip"], z=z, agg=CFG["mit_agg"])
if CFG["arm_filter"]:
    ARMS = [a for a in ARMS if a["name"] in CFG["arm_filter"]]
RES_PATH = OUT_DIR / "results.csv"
J = json.dumps
SECURE_FAMS = {"norm_only", "faithful_prev", "faithful_root"}
OBS_KEYS = ["obs_topo_rho", "obs_topo_b2_rho", "obs_attr_rho", "obs_top3", "obs_nn_hit",
            "obs_collude_topo_rho", "obs_collude_topo_b2_rho", "obs_collude_attr_rho", "obs_collude_top3", "obs_collude_nn_hit", "obs_bit_rho"]
BASE_EXTRA = dict(honest_rejected=np.nan, honest_total=np.nan, byz_rejected=np.nan, byz_total=np.nan, hrej_curve="[]",
                  site_accept="[]", kappa_used=np.nan, clip=np.nan, z=np.nan, agg=np.nan, dp_epsilon=np.nan, **{k: np.nan for k in OBS_KEYS})

def dp_epsilon(z, T, delta):
    """RDP accounting of T Gaussian releases with noise multiplier z (sensitivity = clip bound, client-level add/remove,
    full participation, basic RDP->DP conversion). Covers only the noise on the released aggregate."""
    if not z: return np.nan
    alphas = np.concatenate([np.linspace(1.05, 10, 300), np.linspace(10, 1024, 600)])
    return float(np.min(T * alphas / (2 * z * z) + math.log(1 / delta) / (alphas - 1)))

def spearman(a, b):
    return float(pd.Series(np.asarray(a, float)).corr(pd.Series(np.asarray(b, float)), method="spearman"))

def cosmat(V):
    Vn = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12); return Vn @ Vn.T

def inference_metrics(cos_full, b2_m, hists):
    """topo (full update) and topo_b2 (centred output-bias update): Spearman between update similarity and true class-mix
    similarity over site pairs; nn_hit: attacker's most similar site is among the 3 truly most similar (chance 3/(n-1));
    attr: per-site Spearman(centred output-bias update, centred class mix); top3: most over-represented class in the
    attacker's top-3 (chance 3/C)."""
    n = len(hists); iu = np.triu_indices(n, 1)
    Hn = hists / (np.linalg.norm(hists, axis=1, keepdims=True) + 1e-12); S = Hn @ Hn.T
    Bc = b2_m - b2_m.mean(0); Hc = hists - hists.mean(0); Cb = cosmat(Bc)
    attr = float(np.nanmean([spearman(Bc[i], Hc[i]) for i in range(n)]))
    top3 = float(np.mean([int(np.argmax(Hc[i]) in np.argsort(-Bc[i])[:3]) for i in range(n)]))
    nn = []
    for i in range(n):
        g = Cb[i].copy(); g[i] = -np.inf; t = S[i].copy(); t[i] = -np.inf
        nn.append(int(int(np.argmax(g)) in set(np.argsort(-t)[:3].tolist())))
    return spearman(cos_full[iu], S[iu]), spearman(Cb[iu], S[iu]), attr, top3, float(np.mean(nn))

def final_row(arm, fam, tau, scn, seed, n_byz, w, bundle, curve, extra, t_start):
    Xt, yt = bundle["test"]; m = metrics(w, Xt, yt, full=True)
    return dict(arm=arm, family=fam, tau=tau, scenario=scn, seed=seed, n_byz=n_byz, rounds=CFG["rounds"],
                final_macro_f1=m["macro_f1"], final_bal_acc=m["bal_acc"], final_acc=m["acc"], final_detect=m["detect"],
                final_fpr=m["fpr"], final_asr=asr(w, bundle["test_trig"]),
                last3_macro_f1=float(np.mean(curve[-3:])) if curve else m["macro_f1"],
                per_class_recall=J(m["per_class_recall"]), curve_macro_f1=J(curve),
                site_tv=J([round(x, 4) for x in bundle["site_tv"]]), seconds=round(time.time() - t_start, 1),
                **{**BASE_EXTRA, **extra})

def run_one(arm, scn, seed, bundle):
    t_start = time.time(); fam, tau, name = arm["fam"], arm["tau"], arm["name"]; kap = arm.get("kappa") or KAPPA
    rng = np.random.default_rng(10_000 + seed); crng = np.random.default_rng(20_000 + seed)
    sites, (Xr, yr), (Xt, yt) = bundle["sites"], bundle["root"], bundle["test"]
    n = len(sites); rounds = CFG["rounds"]
    n_byz = 0 if scn == "none" else (3 if scn.endswith("_f30") else CFG["n_byz"]); base = scn.replace("_f30", "")
    byz = set(range(n_byz)); pois = {}
    for k in byz:
        Xs, ys = sites[k]
        if base == "labelflip": pois[k] = (Xs, (ys + 1) % C)
        elif base == "targeted_flip": pois[k] = (Xs, np.where(ys != NORMAL, NORMAL, ys))
        elif base in ("backdoor", "backdoor_boost"):
            m = (rng.random(len(ys)) < CFG["backdoor_frac"]) & (ys != NORMAL)
            Xp = Xs.copy(); Xp[np.ix_(np.where(m)[0], TRIG)] = CFG["trigger_val"]; yp = ys.copy(); yp[m] = NORMAL
            pois[k] = (Xp, yp)
    obs_on = (scn == "none"); n_obs = 0
    if obs_on:
        cos_v = np.zeros((n, n)); b2_v = np.zeros((n, C)); cos_c = np.zeros((n, n)); b2_c = np.zeros((n, C))
    w = init_w(rng); prev = None; acc_site = np.zeros(n); hrej_c = []; hr = hb = ht = bt = 0; curve = []
    for r in range(rounds):
        root_u = local_update(w, Xr, yr, rng); rn = np.linalg.norm(root_u); B = kap * rn
        ref = {"faithful_root": root_u, "faithful_prev": prev}.get(fam)
        ups = []
        for k, (Xs, ys) in enumerate(sites):
            u = local_update(w, *pois[k], rng) if k in pois else local_update(w, Xs, ys, rng)
            if k in byz:
                if base == "signflip": u = -4.0 * u
                elif base == "backdoor_boost": u = CFG["boost"] * u
                elif base == "adaptive": u = adaptive_vec(B, ref, tau, u)
            ups.append(u)
        U = np.array(ups); mask = None; subs = None
        if fam == "fedavg": agg = U.mean(0)
        elif fam == "trimmed": agg = agg_trimmed(U, CFG["trim_k"])
        elif fam == "krum":
            agg, sel = agg_multikrum(U, CFG["krum_f"]); mask = [i in sel for i in range(n)]
        else:
            subs = [fa.make_submission(k, u, crng, with_commitments=False) for k, u in enumerate(ups)]
            res = fa.run_round(subs, crng, B, tau, ref, verify_openings=False)
            agg = res.mean_update; mask = [bool(b) for b in res.bits]
            acc_idx = [k for k, b in enumerate(mask) if b]
            if acc_idx and (arm.get("clip") or arm.get("z")):                # ideal secure clip (+ noise inside the aggregate)
                Ct = (arm.get("clip") or kap) * rn
                agg = np.array([ups[k] * min(1.0, Ct / (np.linalg.norm(ups[k]) + 1e-12)) for k in acc_idx]).mean(0)
                if arm.get("z"): agg = agg + crng.normal(0, arm["z"] * Ct / len(acc_idx), D)
            if arm.get("agg"):                                               # public post-processing of the revealed aggregate
                bound = arm["agg"] * np.abs(root_u) + 0.5 * np.sqrt(np.mean(root_u ** 2)); agg = np.clip(agg, -bound, bound)
            if res.n_accepted: prev = res.mean_update
        if obs_on and r < CFG["obs_rounds"]:
            if subs is not None:                                             # server-1 share view (+ colluding view)
                V = np.array([[x / fa.Q - 0.5 for x in s.shares[0]] for s in subs])
                cos_c += cosmat(U); b2_c += U[:, -C:]
            else:                                                            # aggregator sees plaintext updates
                V = U
            cos_v += cosmat(V); b2_v += V[:, -C:]; n_obs += 1
        if mask is not None:
            h_r = 0
            for k in range(n):
                acc_site[k] += mask[k]
                if k in byz: bt += 1; hb += (not mask[k])
                else: ht += 1; hr += (not mask[k]); h_r += (not mask[k])
            hrej_c.append(h_r)
        w = w + agg
        curve.append(round(metrics(w, Xt, yt)["macro_f1"], 4))
    extra = {}
    if mask is not None:
        extra.update(honest_rejected=hr, honest_total=ht, byz_rejected=hb, byz_total=bt, hrej_curve=J(hrej_c),
                     site_accept=J([int(x) for x in acc_site]))
    if fam in SECURE_FAMS:
        extra.update(kappa_used=kap, clip=arm.get("clip", np.nan) or np.nan, z=arm.get("z", np.nan) or np.nan,
                     agg=arm.get("agg", np.nan) or np.nan, dp_epsilon=dp_epsilon(arm.get("z"), rounds, CFG["dp_delta"]))
    if obs_on and n_obs:
        hists = np.array(bundle["site_hist"])
        t1, t2, a_, k3, nn = inference_metrics(cos_v / n_obs, b2_v / n_obs, hists)
        extra.update(obs_topo_rho=t1, obs_topo_b2_rho=t2, obs_attr_rho=a_, obs_top3=k3, obs_nn_hit=nn)
        if fam in SECURE_FAMS:
            c1, c2, c3, c4, c5 = inference_metrics(cos_c / n_obs, b2_c / n_obs, hists)
            extra.update(obs_collude_topo_rho=c1, obs_collude_topo_b2_rho=c2, obs_collude_attr_rho=c3, obs_collude_top3=c4,
                         obs_collude_nn_hit=c5, obs_bit_rho=spearman(acc_site, bundle["site_tv"]))   # accept bits over ALL rounds
    return final_row(name, fam, tau, scn, seed, n_byz, w, bundle, curve, extra, t_start)

def run_local_only(seed, bundle):
    t0 = time.time(); rng = np.random.default_rng(30_000 + seed); Xt, yt = bundle["test"]; ms = []; ws = []
    for Xs, ys in bundle["sites"]:
        w = init_w(rng)
        for r in range(CFG["rounds"]): w = w + local_update(w, Xs, ys, rng)
        ms.append(metrics(w, Xt, yt)); ws.append(w)
    row = final_row("local_only", "local_only", None, "none", seed, 0, ws[0], bundle, [], {}, t0)
    for k in ("macro_f1", "bal_acc", "acc", "detect", "fpr"):
        row["final_" + k] = float(np.mean([m[k] for m in ms]))
    row["last3_macro_f1"] = row["final_macro_f1"]; row["final_asr"] = float(np.mean([asr(w, bundle["test_trig"]) for w in ws]))
    return row

def run_centralized(seed, bundle):
    t0 = time.time(); rng = np.random.default_rng(40_000 + seed)
    Xp = np.concatenate([s[0] for s in bundle["sites"]]); yp = np.concatenate([s[1] for s in bundle["sites"]])
    w = init_w(rng); curve = []
    for r in range(CFG["rounds"]):
        w = w + local_update(w, Xp, yp, rng, batch=CFG["batch"] * CFG["n_sites"])
        curve.append(round(metrics(w, *bundle["test"])["macro_f1"], 4))
    return final_row("centralized", "centralized", None, "none", seed, 0, w, bundle, curve, {}, t0)

done = set()
if RES_PATH.exists():
    old = pd.read_csv(RES_PATH); done = set(zip(old.arm, old.scenario, old.seed)); print(f"resuming: {len(done)} runs done")
want = lambda name: (not CFG["arm_filter"]) or (name in CFG["arm_filter"])
bundle_info = {}; t0 = time.time(); count = 0
n_total = len(CFG["seeds"]) * (2 + sum(len(a["scn"] if a["scn"] is not None else CFG["scenarios"]) for a in ARMS))
for seed in CFG["seeds"]:
    bundle = build_bundle(seed)
    bundle_info[seed] = {k: bundle[k] for k in ("site_sizes", "site_attack_frac", "site_n_classes", "site_tv")}
    jobs = []
    if want("local_only"): jobs.append(("local_only", None, "none"))
    if want("centralized"): jobs.append(("centralized", None, "none"))
    jobs += [(a["name"], a, s) for a in ARMS for s in (a["scn"] if a["scn"] is not None else CFG["scenarios"])]
    for name, arm, scn in jobs:
        count += 1
        if (name, scn, seed) in done: continue
        row = (run_local_only(seed, bundle) if name == "local_only" else run_centralized(seed, bundle) if name == "centralized"
               else run_one(arm, scn, seed, bundle))
        pd.DataFrame([row]).to_csv(RES_PATH, mode="a", header=not RES_PATH.exists(), index=False)
        print(f"[{count}/{n_total}] seed={seed} {scn:14s} {name:34s} macroF1={row['final_macro_f1']:.3f} ASR={row['final_asr']:.3f} ({time.time() - t0:.0f}s)")
    del bundle
print("done.")

## 5. Summary tables, paired statistics (t-based CIs), privacy, mitigation with epsilon, kappa sweep, fairness

In [ ]:
# ============================== 5. SUMMARY ==============================
import matplotlib.pyplot as plt
res = pd.read_csv(RES_PATH)
res["honest_rej_rate"] = res.honest_rejected / res.honest_total.replace(0, np.nan)
res["byz_rej_rate"] = res.byz_rejected / res.byz_total.replace(0, np.nan)
TT = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262, 10: 2.228}   # t critical values, df = n-1
tc = lambda n: TT.get(n - 1, 2.1 if n <= 31 else 1.96)
ci = lambda x: tc(len(x)) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
pd.set_option("display.width", 240); pd.set_option("display.max_rows", 600); pd.set_option("display.max_columns", 50)
SC_ORDER = ["none", "signflip", "labelflip", "targeted_flip", "backdoor", "backdoor_boost", "adaptive", "adaptive_f30"]
names = [a["name"] for a in ARMS]; order = ["centralized", "local_only"] + names; present = set(res.arm)
print("seeds:", sorted(res.seed.unique()), "| runs:", len(res))

summary = res.groupby(["scenario", "arm"]).agg(n=("seed", "count"), macro_f1=("final_macro_f1", "mean"), f1_ci95=("final_macro_f1", ci),
        bal_acc=("final_bal_acc", "mean"), asr=("final_asr", "mean"), asr_ci95=("final_asr", ci), fpr=("final_fpr", "mean"),
        honest_rej=("honest_rej_rate", "mean"), byz_rej=("byz_rej_rate", "mean")).reset_index()
summary.to_csv(OUT_DIR / "summary.csv", index=False)

fmt = lambda m, c: f"{m:.3f} +/- {c:.3f}" if pd.notna(c) else f"{m:.3f}"
hl = summary.assign(cell=[fmt(m, c) for m, c in zip(summary.macro_f1, summary.f1_ci95)]).pivot(index="arm", columns="scenario", values="cell")
hl = hl.reindex([a for a in order if a in hl.index])[[s for s in SC_ORDER if s in hl.columns]]; hl.to_csv(OUT_DIR / "headline_macro_f1.csv")
print("\nMACRO-F1 (mean +/- t95 CI over seeds)"); print(hl.to_string())
ha = summary[summary.scenario.isin(["backdoor", "backdoor_boost"])].assign(cell=lambda d: [fmt(m, c) for m, c in zip(d.asr, d.asr_ci95)]).pivot(index="arm", columns="scenario", values="cell")
ha = ha.reindex([a for a in order if a in ha.index]); ha.to_csv(OUT_DIR / "headline_asr.csv")
print("\nBACKDOOR ATTACK SUCCESS RATE (lower is better)"); print(ha.to_string())
rj = summary.pivot_table(index="arm", columns="scenario", values=["honest_rej", "byz_rej"]).reindex([a for a in order if a in present]).dropna(how="all")
rj.to_csv(OUT_DIR / "rejection.csv"); print("\nREJECTION RATES (honest / attacker)"); print(rj.round(3).to_string())

def paired(a, b, scn, col="final_macro_f1"):
    x = res[(res.arm == a) & (res.scenario == scn)].set_index("seed")[col]; y = res[(res.arm == b) & (res.scenario == scn)].set_index("seed")[col]
    d = (x - y).dropna(); n = len(d)
    return dict(A=a, B=b, scenario=scn, metric=col, mean_diff=d.mean(), ci95_t=ci(d) if n > 1 else np.nan, wins=int((d > 0).sum()), n=n)
rows = []
for scn in sorted(set(res.scenario), key=lambda s: SC_ORDER.index(s) if s in SC_ORDER else 99):
    for a in [x for x in names if x.startswith("faithful")]:
        for b in ("fedavg", "secure_norm_only", "multi_krum", "trimmed_mean", "faithful_root_t-0.3"):
            if a != b and a in present and b in present and not res[(res.arm == a) & (res.scenario == scn)].empty and not res[(res.arm == b) & (res.scenario == scn)].empty:
                rows.append(paired(a, b, scn))
                if scn in ("backdoor", "backdoor_boost"): rows.append(paired(a, b, scn, "final_asr"))
for scn in ("none",):
    for a, b in (("fedavg", "local_only"), ("centralized", "fedavg")):
        if a in present and b in present: rows.append(paired(a, b, scn))
pairs = pd.DataFrame(rows); pairs.to_csv(OUT_DIR / "paired.csv", index=False)
KEY = pairs[pairs.A.isin(["faithful_root_t-0.3", "fedavg", "centralized"]) & pairs.B.isin(["fedavg", "secure_norm_only", "multi_krum", "trimmed_mean", "local_only"])]
print("\nPAIRED (A - B), recommended design vs baselines:"); print(KEY.round(3).to_string(index=False))

# ---- privacy
pv = res[(res.scenario == "none") & res.obs_attr_rho.notna()]
if len(pv):
    out = []
    for arm, g in pv.groupby("arm"):
        d = {"arm": arm, "view": "server-1 share + accept bits" if g.family.iloc[0] in SECURE_FAMS else "plaintext updates"}
        for c in OBS_KEYS:
            d[c] = g[c].mean(); d[c + "_ci"] = ci(g[c].dropna()) if g[c].notna().sum() > 1 else np.nan
        out.append(d)
    priv = pd.DataFrame(out); priv.to_csv(OUT_DIR / "privacy.csv", index=False)
    chance = dict(topo=0.0, attr=0.0, top3=3 / C, nn=3 / (CFG["n_sites"] - 1))
    print(f"\nPRIVACY (chance: correlations 0, top-3 class hit {chance['top3']:.2f}, nearest-neighbour hit {chance['nn']:.2f})")
    cols = ["arm", "view", "obs_attr_rho", "obs_top3", "obs_topo_rho", "obs_topo_b2_rho", "obs_nn_hit", "obs_collude_attr_rho", "obs_collude_top3", "obs_collude_nn_hit", "obs_bit_rho"]
    print(priv[cols].round(3).to_string(index=False))
    pr = []
    for col in ["obs_attr_rho", "obs_top3", "obs_topo_rho", "obs_topo_b2_rho", "obs_nn_hit"]:
        for a in ("fedavg", "multi_krum", "trimmed_mean"):
            if a in present and "faithful_root_t-0.3" in present: pr.append(paired(a, "faithful_root_t-0.3", "none", col))
    ppair = pd.DataFrame(pr); ppair.to_csv(OUT_DIR / "privacy_paired.csv", index=False)
    print("\nPRIVACY PAIRED (plaintext view minus share view; positive = plaintext leaks more)"); print(ppair.round(3).to_string(index=False))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True); p = priv.set_index("arm")
    for ax, (c, ttl, ch) in zip(axes, [("obs_attr_rho", "class mix recovered (Spearman)", 0.0), ("obs_top3", "dominant class in top-3 (hit rate)", chance["top3"]),
                                       ("obs_nn_hit", "most-similar-site identification (hit rate)", chance["nn"])]):
        ax.barh(p.index, p[c], xerr=p[c + "_ci"].fillna(0), color=["#E45756" if v == "plaintext updates" else "#54A24B" for v in p.view])
        ax.axvline(ch, color="k", ls="--", lw=1); ax.set_title(ttl, fontsize=10)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "privacy_attack.png", dpi=130); plt.show()

# ---- mitigation (with epsilon)
mit = [a for a in order if a.startswith("faithful_root_t-0.3") and a in present and ("_clip" in a or "_all" in a or a == "faithful_root_t-0.3")]
mit += [a for a in ("trimmed_mean", "multi_krum", "fedavg") if a in present]
ms_ = summary[summary.arm.isin(mit)]
if len(ms_):
    eps = res.groupby("arm").dp_epsilon.first()
    mt = ms_.pivot_table(index="arm", columns="scenario", values=["macro_f1", "asr", "honest_rej"]).reindex(mit)
    mt[("epsilon", "RDP, delta=%g" % CFG["dp_delta"])] = [eps.get(a, np.nan) for a in mt.index]
    mt.to_csv(OUT_DIR / "mitigation.csv"); print("\nMITIGATION (macro-F1, ASR, honest rejection; epsilon = noise-only RDP accounting over %d rounds)" % CFG["rounds"]); print(mt.round(3).to_string())
    if {"none", "backdoor"} <= set(ms_.scenario):
        a_ = ms_[ms_.scenario == "none"].set_index("arm").macro_f1; b_ = ms_[ms_.scenario == "backdoor"].set_index("arm").asr
        plt.figure(figsize=(7, 5))
        for a in mit:
            if a in a_.index and a in b_.index:
                plt.scatter(a_[a], b_[a]); plt.annotate(a.replace("faithful_root_t-0.3", "F"), (a_[a], b_[a]), fontsize=7)
        plt.xlabel("clean macro-F1 (higher is better)"); plt.ylabel("backdoor attack success rate (lower is better)")
        plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "mitigation_tradeoff.png", dpi=130); plt.show()

# ---- kappa sweep
kn = [a for a in present if a == "faithful_root_t-0.3" or a.startswith("faithful_root_t-0.3_kappa")]
if len(kn) > 1:
    ks = summary[summary.arm.isin(kn) & summary.scenario.isin(["none", "signflip", "adaptive_f30"])]
    kt = ks.pivot_table(index="arm", columns="scenario", values=["macro_f1", "honest_rej"]); kt.to_csv(OUT_DIR / "kappa.csv")
    print(f"\nKAPPA SWEEP (base arm uses kappa = {KAPPA}; sweep arms are named by kappa)"); print(kt.round(3).to_string())

# ---- fairness
fr = []
for _, r in res[(res.scenario == "none") & (res.site_accept != "[]")].iterrows():
    acc_ = json.loads(r.site_accept); tv = json.loads(r.site_tv)
    for k in range(len(acc_)): fr.append(dict(arm=r.arm, seed=r.seed, site=k, accept_rate=acc_[k] / r.rounds, tv=tv[k]))
fair = pd.DataFrame(fr)
if len(fair):
    ft = fair.groupby("arm").apply(lambda g: pd.Series(dict(spearman_accept_vs_tv=g.accept_rate.corr(g.tv, method="spearman"),
            accept_lowTV=g[g.tv <= g.tv.quantile(1 / 3)].accept_rate.mean(), accept_highTV=g[g.tv >= g.tv.quantile(2 / 3)].accept_rate.mean()))).reset_index()
    ft.to_csv(OUT_DIR / "fairness.csv", index=False); print("\nFAIRNESS (no attack): acceptance vs distance of a site's class mix from the root distribution"); print(ft.round(3).to_string(index=False))

scns = sorted(set(res.scenario), key=lambda s: SC_ORDER.index(s) if s in SC_ORDER else 99)
fig, axes = plt.subplots(1, len(scns), figsize=(3.8 * len(scns), 8), sharex=True); axes = np.atleast_1d(axes)
for ax, scn in zip(axes, scns):
    s = summary[summary.scenario == scn].set_index("arm"); s = s.reindex([a for a in order if a in s.index])
    ax.barh(s.index, s.macro_f1, xerr=s.f1_ci95.fillna(0), color="#4C78A8"); ax.set_title(scn); ax.set_xlim(0, 1); ax.set_xlabel("macro-F1")
plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "macro_f1_by_arm.png", dpi=130); plt.show()

## 6. Cost benchmark with REAL commitments, Merkle inclusion and audit

In [ ]:
# ============================== 6. COST BENCHMARK ==============================
b0 = build_bundle(0); rb = np.random.default_rng(6); rc = np.random.default_rng(5)
w0 = init_w(rb); ups = [local_update(w0, Xs, ys, rb) for Xs, ys in b0["sites"]]
root_u = local_update(w0, *b0["root"], rb); Bq = KAPPA * np.linalg.norm(root_u)
t = time.perf_counter(); subs = [fa.make_submission(k, u, rc, with_commitments=True) for k, u in enumerate(ups)]; t_client = time.perf_counter() - t
led = fa.Ledger(); t = time.perf_counter()
res_c = fa.run_round(subs, rc, Bq, tau=0.2, ref=root_u, verify_openings=True, ledger=led); t_round = time.perf_counter() - t
t = time.perf_counter(); audit_ok = fa.audit(res_c.transcript); t_audit = time.perf_counter() - t
incl_ok = all(fa.site_verify(res_c.transcript, i, j)[0] for i in range(len(subs)) for j in range(2))
n = len(subs); placeholder_cmp = n * 3 * 2 * fa.CMP_ELEMS * fa.FE_BYTES; plain_bytes = n * D * 4
cost = dict(model_dim=D, n_sites=n, audit_ok=bool(audit_ok), inclusion_proofs_ok=bool(incl_ok), accepted=int(res_c.n_accepted),
            bytes_by_phase={k: int(v) for k, v in led.bytes_.items()}, secs_by_phase={k: round(v, 3) for k, v in led.secs.items()},
            client_commit_and_share_secs_total=round(t_client, 3), server_round_secs=round(t_round, 3), audit_secs=round(t_audit, 3),
            total_online_bytes=int(led.total_bytes()), placeholder_comparison_bytes=int(placeholder_cmp),
            online_bytes_excluding_placeholder=int(led.total_bytes() - placeholder_cmp),
            plain_fedavg_upload_bytes_float32=int(plain_bytes), ratio_vs_plain=round(led.total_bytes() / plain_bytes, 2),
            ratio_vs_plain_excluding_placeholder=round((led.total_bytes() - placeholder_cmp) / plain_bytes, 2),
            offline_triple_bytes=int(led.bytes_.get("offline", 0)),
            ideal_clip_extra_online_bytes_estimate=int(n * 2 * D * fa.FE_BYTES))   # one extra masked-vector opening per site (estimate)
cost["ratio_vs_plain_with_ideal_clip"] = round((led.total_bytes() + cost["ideal_clip_extra_online_bytes_estimate"]) / plain_bytes, 2)
json.dump(cost, open(OUT_DIR / "cost.json", "w"), indent=2); print(json.dumps(cost, indent=1))

## 7. Export (aggregates only, no raw data)

In [ ]:
# ============================== 7. EXPORT ==============================
audit_info = dict(file_rows=FILE_ROWS, file_class_counts=FILE_COUNTS, classes=CLASSES, sampled_rows=int(len(y_arr)),
                  sampled_attack_share=float(y_bin.mean()), sampled_class_counts=pd.Series(types_arr).value_counts().to_dict(),
                  n_features=len(FEATURES), feature_names=FEATURES, feature_groups_before_selection=groups,
                  skipped_columns=skipped, model_dim=D, kappa=KAPPA, partitions=bundle_info)
json.dump(audit_info, open(OUT_DIR / "data_audit.json", "w"), indent=1)
env = dict(python=sys.version, platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__,
           cpu_count=os.cpu_count(), finished=datetime.datetime.now().isoformat(timespec="seconds"))
json.dump(env, open(OUT_DIR / "env.json", "w"), indent=1)
zip_path = OUT_DIR / "results_pack.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["config.json", "results.csv", "summary.csv", "paired.csv", "fairness.csv", "privacy.csv", "privacy_paired.csv", "mitigation.csv", "kappa.csv", "headline_macro_f1.csv", "headline_asr.csv", "rejection.csv", "cost.json", "data_audit.json", "env.json"]:
        if (OUT_DIR / name).exists(): z.write(OUT_DIR / name, name)
    for p in (OUT_DIR / "figs").glob("*.png"): z.write(p, f"figs/{p.name}")
print(f"wrote {zip_path.resolve()} ({zip_path.stat().st_size / 1024:.0f} KiB). Upload it to the chat (aggregates only).")